# Natural language to SQL





## Configure Hugging Face token

To run this lab, first you'll need to get a Hugging Face token. Follow these steps:

1. Create a free account at [https://huggingface.co](https://huggingface.co/)
2. Go to Settings → Access Tokens
3. Generate a token (read access is enough)

Then, you'll need to upload this notebook to Google Colab and configure the token you've just created:

1. Go to [https://colab.research.google.com](https://colab.research.google.com)
2. Upload this notebook (e.g., File → Upload notebook)
3. Click on "🔑 Secrets" (left panel)
4. Select "Add new secret":
    - Name: HF_TOKEN
    - Value: paste your Hugging Face token

Note:
- Run this notebook directly in the Google Colab browser interface (https://colab.research.google.com/).
- Other environments (such as VS Code with the Google Colab extension) may not support Colab Secrets, which means your HF_TOKEN may not be accessible and the notebook may fail to run as expected.


In [1]:
from google.colab import userdata
userdata.get('HF_TOKEN')

ModuleNotFoundError: No module named 'google.colab'

In [3]:
# Check if Hugging Face token is loaded correctly

try:
    token = userdata.get("HF_TOKEN")

    if token:
        print("HF_TOKEN found. You're ready to continue! ✅")
    else:
        print(
            "HF_TOKEN is not available ❌ \n"
            "Please follow the setup instructions above to add your Hugging Face "
            "token as a Colab Secret named 'HF_TOKEN'."
        )
except Exception:
    print(
        "Unable to access 'HF_TOKEN' ❌ \n"
        "Make sure you're running this notebook in the Google Colab web interface "
        "and that you've added a Colab Secret named 'HF_TOKEN'."
    )

HF_TOKEN found. You're ready to continue! ✅


## Initial Setup

In [4]:
#Install the lastest versions of peft & transformers library recommended
#if you want to work with the most recent models
!pip install -q git+https://github.com/huggingface/peft.git
!pip install git+https://github.com/huggingface/accelerate.git
!pip install git+https://github.com/huggingface/transformers.git
!pip install bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/huggingface/accelerate.git to /tmp/pip-req-build-p91yuo0c
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/accelerate.git /tmp/pip-req-build-p91yuo0c
  Resolved https://github.com/huggingface/accelerate.git to commit fd132822ff0aea945786a263a49e6730249917cb
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for accelerate: filename=accelerate-1.15.0.dev0-py3-none-any.whl size=392895 sha256=00543d606c93b9327c842e16ac7bff806ffcc51d6b2638efb48c4764430d72e9
  Stored in directory: /tmp/pip-ephem-wheel-cache-gauwsedk/wheels/b0/31/a9/2d1286ef99972e43925a21317d7a0255852c3efe5e738f5a69
Successfully built accelerate
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0

In [5]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
import accelerate

In [6]:
model_name = "defog/sqlcoder-7b"

`defog/sqlcoder-7b` is based on the Mistral 7B architecture and has been fine-tuned for text-to-SQL tasks, enabling it to generate high-quality SQL queries from natural language.


We need to create the Quantization configuration to load the Model.

It is a large model and I want it to fit in a 16GB GPU, I'm going to use a 4 bits quantization.

If you want to learn more about quantization, refer to this article: [QLoRA: Training a Large Language Model on a 16GB GPU.](https://medium.com/towards-artificial-intelligence/qlora-training-a-large-language-model-on-a-16gb-gpu-00ea965667c1)

You can try to use this model in a 8 bit quantizations and check in you see any improvements in the results.

In [7]:
bnb_config = BitsAndBytesConfig(
  load_in_4bit=True,
  bnb_4bit_use_double_quant=True,
  bnb_4bit_quant_type="nf4",
  bnb_4bit_compute_dtype=torch.bfloat16
)


To load the model I pass to the AutoModelForCasualLM teh quantization configurations, and HuggingFace take care of all the hard work.

In [ ]:
foundation_model = AutoModelForCausalLM.from_pretrained(model_name,
                    quantization_config=bnb_config,
                    device_map='auto',
                    use_cache = True)

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

pytorch_model.bin.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
eos_token_id = tokenizer.convert_tokens_to_ids(["```"])[0]

This function wraps the call to *model.generate*

In [ ]:
#this function returns the outputs from the model received, and inputs.
def get_outputs(model, inputs, max_new_tokens=400):
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        num_return_sequences=1,
        eos_token_id=eos_token_id,
        pad_token_id=eos_token_id,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=5
    )
    return outputs

# Prompt without Shots.
In this first PROMPT we are going to give Instructions to the model and pass the structure of the Database.

The instructions are significantly different from those we are passing to GPT-3.5-Turbo. This model is really well fine-tuned, but it is smaller than GPT-3.5.

We need to be more clear with the instructions, as it does not have the same capacity to understand our orders as GPT-3.5.

In [ ]:
sp_nl2sql = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question

    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE TABLE employees (
        ID_Usr INT PRIMARY KEY,
        name VARCHAR(100),
        department VARCHAR(100)
    );

    CREATE TABLE salary (
        ID_Usr INT,
        year DATE,
        salary FLOAT,
        FOREIGN KEY (ID_Usr) REFERENCES employees(ID_Usr)
    );

    CREATE TABLE studies (
        ID INT PRIMARY KEY,
        ID_Usr INT,
        educational_level INT,
        Institution VARCHAR(200),
        Years DATE,
        Speciality VARCHAR(200),
        FOREIGN KEY (ID_Usr) REFERENCES employees(ID_Usr)
    );

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `{question}`:
````````sql3
    """

In [ ]:
sp_nl2sql = sp_nl2sql.format(question="YOUR QUERY HERE")
print(sp_nl2sql)

In [ ]:
input_sentences = tokenizer(sp_nl2sql, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)

In [ ]:
#Empty the cache in orde to do more calls without problems.
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

The SQL Order is correct.

#Prompt with shots OpenAI Style.
In this second prompt we are going to add some Shots with samples to see if our SQL style affects the model.

In [ ]:
sp_nl2sql2 = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to clearn more about teh Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE TABLE employees (
        ID_Usr INT PRIMARY KEY,
        name VARCHAR(100),
        department VARCHAR(100)
    );
    CREATE TABLE salary (
        ID_Usr INT,
        year DATE,
        salary FLOAT,
        FOREIGN KEY (ID_Usr) REFERENCES employees(ID_Usr)
    );
    CREATE TABLE studies (
        ID INT PRIMARY KEY,
        ID_Usr INT,
        educational_level INT,
        Institution VARCHAR(200),
        Years DATE,
        Speciality VARCHAR(200),
        FOREIGN KEY (ID_Usr) REFERENCES employees(ID_Usr)
    );

    ### Response
    Q: How many employees are there?
    A: SELECT COUNT(*) FROM employees;

    Q: What is the average salary in 2023?
    A: SELECT AVG(salary) FROM salary WHERE year = '2023-01-01';

    Q: List employees and their speciality.
    A: SELECT e.name, st.Speciality FROM employees e JOIN studies st ON e.ID_Usr = st.ID_Usr;

    `{question}`:
```````sql3
    """


In [ ]:
sp_nl2sql2 = sp_nl2sql2.format(question="Return The name of the best paid employee")
(print(sp_nl2sql2))

In [ ]:
input_sentences = tokenizer(sp_nl2sql2, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

The Order is really different from the one obtained with the first prompt.

The first difference is the format. But The SQL is realy more simple, at least it is my sensation.

#Prompt with Shots in Sample Style.

In this prompt, we will place the examples in a separate section, and in the instructions, we will instruct the model to pay attention to them in order to generate the SQL commands.

In [ ]:
sp_nl2sql3b = """
    ### Instructions:
Your task is convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- **Deliberately go through the question and database schema word by word** to appropriately answer the question
- **Use the samples SQL In the ### Samples section to learn more about the Databases structure


    ### Input
    Generate a SQL query that answers the question below.
    This query will run on a database whose schema is represented in this string:

    CREATE TABLE employees (
        ID_Usr INT PRIMARY KEY,
        name VARCHAR(100),
        department VARCHAR(100)
    );
    CREATE TABLE salary (
        ID_Usr INT,
        year DATE,
        salary FLOAT,
        FOREIGN KEY (ID_Usr) REFERENCES employees(ID_Usr)
    );
    CREATE TABLE studies (
        ID INT PRIMARY KEY,
        ID_Usr INT,
        educational_level INT,
        Institution VARCHAR(200),
        Years DATE,
        Speciality VARCHAR(200),
        FOREIGN KEY (ID_Usr) REFERENCES employees(ID_Usr)
    );

    ### Samples

    Q: How many employees are there?
    A: SELECT COUNT(*) FROM employees;

    Q: What is the average salary in 2023?
    A: SELECT AVG(salary) FROM salary WHERE year = '2023-01-01';

    Q: List employees and their speciality.
    A: SELECT e.name, st.Speciality FROM employees e JOIN studies st ON e.ID_Usr = st.ID_Usr;

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `{question}`:
``````sql3
    """

In [ ]:
sp_nl2sql3 = sp_nl2sql3b.format(question="Return The name of the best paid employee")
print (sp_nl2sql3)

In [ ]:
input_sentences = tokenizer(sp_nl2sql3, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

#Now the question in spanish.


In [ ]:
sp_nl2sql3 = sp_nl2sql3b.format(question="YOUR QUERY HERE")
print (sp_nl2sql3)

In [ ]:
input_sentences = tokenizer(sp_nl2sql3, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()

In [ ]:
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";")

The generated SQL command is the same regardless of where we have placed the examples.

#Conclusions.

Let's see the three SQL's together.

* SELECT employees.name, MAX(salary.salary) AS max_salary FROM employees JOIN salary ON employees.ID_Usr = salary.ID_Usr GROUP BY employees.name ORDER BY max_salary DESC NULLS LAST LIMIT 1;

* SELECT e.name
    FROM employees e
    JOIN salary s ON e.ID_Usr = s.ID_usr
    WHERE s.salary = (SELECT MAX(salary) FROM salary);

* SELECT e.name
    FROM employees e
    JOIN salary s ON e.ID_Usr = s.ID_usr
    WHERE s.salary = (SELECT MAX(salary) FROM salary);

* Spanish Question: SELECT e.name
     FROM employees e
     JOIN salary s ON e.ID_Usr = s.ID_Usr
     WHERE s.salary = (SELECT MAX(salary) FROM salary)
     GROUP BY e.name
     ORDER BY COUNT(studies.ID_study) DESC
     LIMIT 1;


**The model has demonstrated that it is highly efficient in crafting SQL.** Additionally, it pays a lot of attention, perhaps too much, to the examples we provide. Clearly, these examples should be crafted by one of the best SQL programmers we have access to, though their use may not be essential.

On the other hand, although the model is clearly very proficient in SQL generation, during the creation of the notebook, I have encountered several issues because the commands need to be extremely clear. It doesn't handle typos well (which should not exist).

It appears to have some issues when it receives commands in Spanish. I assume this problem would be present in any language other than English. Therefore, since it's a tool that could be used by non-technical personnel, this should be considered in environments where English is not the primary language.

# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [ ]:
**Version 4 — Chain-of-thought prompt**
``````python
# Version 4: Chain-of-thought — ask the model to reason step by step before writing SQL
sp_nl2sql_cot = """
    ### Instructions:
Your task is to convert a question into a SQL query, given a SQL database schema.
Adhere to these rules:
- First, think step by step about which tables and columns are needed.
- Then write the SQL query.

    ### Input
    Database schema:

    CREATE TABLE employees (
        ID_Usr INT PRIMARY KEY,
        name VARCHAR(100),
        department VARCHAR(100)
    );
    CREATE TABLE salary (
        ID_Usr INT,
        year DATE,
        salary FLOAT,
        FOREIGN KEY (ID_Usr) REFERENCES employees(ID_Usr)
    );
    CREATE TABLE studies (
        ID INT PRIMARY KEY,
        ID_Usr INT,
        educational_level INT,
        Institution VARCHAR(200),
        Years DATE,
        Speciality VARCHAR(200),
        FOREIGN KEY (ID_Usr) REFERENCES employees(ID_Usr)
    );

    ### Response
    Based on your instructions, here is the SQL query I have generated to answer the question
    `{question}`:
`````sql3
    """

sp_nl2sql_cot_filled = sp_nl2sql_cot.format(
    question="Which department has the highest average salary?"
)
input_sentences = tokenizer(sp_nl2sql_cot_filled, return_tensors="pt").to('cuda')
response = get_outputs(foundation_model, input_sentences, max_new_tokens=400)
SQL = tokenizer.batch_decode(response, skip_special_tokens=True)
torch.cuda.empty_cache()
print("Version 4 (Chain-of-thought):")
print(SQL[0].split("```sql3")[-1].split("```")[0].split(";")[0].strip() + ";"

### Key Learnings
1. **Fine-tuning matters more than prompting for basic queries** — sqlcoder-7b generates correct SQL even with minimal prompts, because it was specifically trained for this task.
2. **The `### Samples` section placement outperforms mixing examples into the response** — it keeps the output format clean and the model's attention on the schema.
3. **The model is English-centric** — non-English questions degrade quality noticeably.
4. **Bad few-shot examples didn't fool the model** — the fine-tuning is robust enough to override poor in-context examples, which is not guaranteed with general-purpose models.
5. **4-bit quantization is practical** — the model ran on a 16GB GPU with no visible quality loss for SQL generation tasks.
```